# AOS Pipeline Runner

**Author:** Aaron Roodman  
**Date Created:** 2026-04-12  
**Last Modified:** 2026-04-12  
**Status:** In Progress  
**Keywords:** AOS, pipeline, mktable, DZ fitting, plots  

## Description

Notebook interface for the `run_pipeline.py` pipeline tracker.
Reads `param_sets.yaml` and `runs.yaml`, executes steps, and updates status.

Key functionality:
1. Show status of all pipeline runs
2. Execute pending steps: mktable -> fit -> plots
3. Capture per-step output to `output/{run}_{step}.log`

**Output:** HDF5 donut+visits tables, fit parquet files, plot PDFs, per-step logs

**Based on:** `code/run_pipeline.py` CLI, `param_sets.yaml`, `runs.yaml`

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-12 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Status](#status)
4. [Run Steps](#run)
5. [Manual Status Updates](#manual)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Which run and step to execute
run_name = 'chunk1'    # or None for all runs
step_name = 'mktable'  # 'mktable', 'fit', 'plots', or None for all
dry_run = True          # set False to actually execute

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import io
import contextlib
import subprocess
import sys
import time
from datetime import datetime

sys.path.insert(0, 'code')
from run_pipeline import (
    load_runs, save_runs, load_param_sets, resolve_run,
    hdf5_path, fits_path, step_log_path,
    cmd_status, cmd_run, cmd_set, cmd_reset,
    STEP_ORDER, LOG_FILE, log,
)
from intrinsics_lib import run_mktable
from dz_fitting import run_double_zernike_fits

print('Pipeline tools loaded.')

<a id='status'></a>
## 3. Status

In [ ]:
data = load_runs()
cmd_status(data)

<a id='run'></a>
## 4. Run Steps

Executes pending pipeline steps based on the parameters cell above.

- `run_name = None` → all runs
- `step_name = None` → all pending steps (respecting prerequisites)
- `dry_run = True` → show commands without executing

Output is captured to `output/{run}_{step}.log` and also printed here.

In [ ]:
data = load_runs()
param_sets = load_param_sets()

if dry_run:
    cmd_run(data, run_name, step_name, dry_run=True)
else:
    runs = data.get('runs', {})
    targets = {run_name: runs[run_name]} if run_name else runs

    for name, cfg in targets.items():
        resolved = resolve_run(cfg, param_sets)
        steps = cfg.get('steps', {})

        for step in STEP_ORDER:
            if step_name and step != step_name:
                continue
            status = steps.get(step, 'pending')
            if status not in ('pending', 'failed'):
                continue

            # Check prerequisites
            step_idx = STEP_ORDER.index(step)
            prereqs_ok = all(
                steps.get(s, 'pending') == 'done'
                for s in STEP_ORDER[:step_idx])
            if not prereqs_ok:
                print(f'{name}.{step}: prerequisites not met, skipping')
                continue

            print(f'\n{"="*60}')
            print(f'{name}.{step}: STARTING')
            print(f'{"="*60}')

            data['runs'][name]['steps'][step] = 'running'
            save_runs(data)

            step_logfile = step_log_path(name, step)
            LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
            t0 = time.time()

            try:
                if step == 'mktable':
                    mktable_params = dict(resolved)
                    for k in ['param_set', 'no_single_image',
                              'no_fit_params', 'no_trio']:
                        mktable_params.pop(k, None)
                    mktable_params.setdefault('output_dir', 'output')
                    mktable_params.setdefault('include_thermal', True)
                    mktable_params.setdefault('calc_intrinsics', True)

                    capture = io.StringIO()
                    with contextlib.redirect_stdout(capture):
                        result = await run_mktable(**mktable_params)
                    output_text = capture.getvalue()
                    print(output_text)
                    with open(step_logfile, 'w') as f:
                        f.write(f'# {name}.{step}\n')
                        f.write(f'# {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n\n')
                        f.write(output_text)

                elif step == 'fit':
                    h5 = hdf5_path(resolved)
                    coord = resolved.get('coord_sys', 'OCS')

                    capture = io.StringIO()
                    with contextlib.redirect_stdout(capture):
                        run_double_zernike_fits(input_file=h5, coord_sys=coord)
                    output_text = capture.getvalue()
                    print(output_text)
                    with open(step_logfile, 'w') as f:
                        f.write(f'# {name}.{step}\n')
                        f.write(f'# {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n\n')
                        f.write(output_text)

                elif step == 'plots':
                    h5 = hdf5_path(resolved)
                    fp = fits_path(resolved)
                    coord = resolved.get('coord_sys', 'OCS')
                    cmd = ['python', 'code/run_dz_plots.py', h5, fp,
                           '--coord-sys', coord]
                    for flag in ['no_single_image', 'no_fit_params', 'no_trio']:
                        if resolved.get(flag):
                            cmd.append(f'--{flag.replace("_", "-")}')

                    with open(step_logfile, 'w') as f:
                        f.write(f'# {name}.{step}\n')
                        f.write(f'# {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
                        f.write(f'# {" ".join(cmd)}\n\n')
                        f.flush()
                        proc = subprocess.Popen(
                            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
                        for line in proc.stdout:
                            sys.stdout.write(line)
                            f.write(line)
                        proc.wait()
                    if proc.returncode != 0:
                        raise RuntimeError(f'plots exited with {proc.returncode}')

                elapsed = time.time() - t0
                data['runs'][name]['steps'][step] = 'done'
                save_runs(data)
                log(f'{name}.{step}: DONE ({elapsed:.0f}s) — log: {step_logfile}')

            except Exception as e:
                elapsed = time.time() - t0
                data['runs'][name]['steps'][step] = 'failed'
                save_runs(data)
                log(f'{name}.{step}: FAILED ({elapsed:.0f}s) — {e}')
                print(f'Log: {step_logfile}')
                if not step_name:
                    print(f'Stopping {name} after failed step.')
                    break

<a id='manual'></a>
## 5. Manual Status Updates

Use these cells to manually set or reset step status.

In [ ]:
# Manually mark a step as done (or pending, failed, skip)
# cmd_set(load_runs(), 'chunk1', 'mktable', 'done')

In [ ]:
# Reset all steps for a run
# cmd_reset(load_runs(), 'chunk1')

In [ ]:
# Refresh status after changes
data = load_runs()
cmd_status(data)